In [6]:
# Ozone (O3) geometry in Angstroms
ozone_geometry = (
    "O 0.0000  0.0000  0.0000; "  # Central oxygen at the origin
    "O 0.0000  1.0775  0.6865; "  # Second oxygen
    "O 0.0000 -1.0775  0.6865"   # Third oxygen
)

In [7]:
from qiskit_nature.second_q.drivers import PySCFDriver
driver = PySCFDriver(atom=ozone_geometry, basis="sto-3g")
raw_problem = driver.run()

In [8]:
from qiskit_nature.second_q.mappers import JordanWignerMapper
mapper = JordanWignerMapper()

In [9]:
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
transformer = ActiveSpaceTransformer(num_electrons=2, num_spatial_orbitals=2)
problem = transformer.transform(raw_problem)

In [10]:
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
ansatz = UCCSD(
    problem.num_spatial_orbitals,
    problem.num_particles,
    mapper,
    initial_state=HartreeFock(
        problem.num_spatial_orbitals,
        problem.num_particles,
        mapper,
    ),
)

In [11]:
import numpy as np
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit.primitives import StatevectorEstimator
vqe = VQE(StatevectorEstimator(), ansatz, SLSQP())
vqe.initial_point = np.zeros(ansatz.num_parameters)

In [12]:
from qiskit_algorithms import AdaptVQE
adapt_vqe = AdaptVQE(vqe)
adapt_vqe.supports_aux_operators = lambda: True  # temporary fix? <-

In [13]:
from qiskit_nature.second_q.algorithms import GroundStateEigensolver
solver = GroundStateEigensolver(mapper, adapt_vqe)

In [15]:
result = solver.solve(problem)

print(f"Total ground state energy = {result.total_energies[0]:.4f} Hartree")

Total ground state energy = -221.3894 Hartree
